# CPP8 CUT&Tag — Replication Timing Enrichment

Strategy:
1. Predict genome-wide signals (G1, ES, MS, LS) with the trained model.
2. Classify each 128-bp bin: divide ES/MS/LS by G1; the dominant phase (ratio ≥ 1) wins; if all < G1 → non-replicating.
3. **Validate** predicted classification against true labels (per-chromosome accuracy + confusion matrix).
4. Overlap CPP8 CUT&Tag peaks with classified bins.
5. Test whether CPP8 peaks are enriched in early-replicating (ES) bins.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from scipy.stats import chi2_contingency, mannwhitneyu
from pathlib import Path
from pyfaidx import Fasta

from src.infer._utils import load_model, rc_average
from src.data.data_utils import load_signals, one_hot_encode

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
CHECKPOINT  = 'outputs/checkpoints/best_model_0518.pt'
CONFIG      = 'src/configs/transformer_wt.yaml'
FASTA       = '/liaozizhuo/repli-ATAC-seq/data/genomes/rice_all_genomes_v7.fasta'
LABELS_TSV  = '/liaozizhuo/repli-ATAC-seq/data/labels/rice_128bp_rt_signals.tsv'
SPECIES     = 'rice'

CPP8_BED    = None  # <-- FILL IN: path to CPP8 CUT&Tag peaks BED file

# If a previous run saved bins.csv, set this path to skip re-prediction
BINS_CSV    = None  # e.g. 'outputs/cpp8_enrichment/bins.csv'

CHROMS      = [f'chr{i:02d}' for i in range(1, 13)]
RC          = True
BATCH_SIZE  = 16
OUTPUT_DIR  = 'outputs/cpp8_enrichment'

# Genome geometry — must match training config
BIN_SIZE    = 128
CROP_BINS   = 320
CROP_BP     = CROP_BINS * BIN_SIZE   # 40960 bp
OUT_BINS    = 896
STRIDE      = OUT_BINS * BIN_SIZE    # 114688 bp — gapless tiling

PHASE_ORDER = ['ES', 'MS', 'LS', 'non-rep']
PHASE_NAMES = np.array(['ES', 'MS', 'LS'])
COLORS      = {'ES': '#d62728', 'MS': '#ff7f0e', 'LS': '#1f77b4', 'non-rep': '#b0b0b0'}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
model, cfg = load_model(CHECKPOINT, CONFIG, device)
window_size = cfg['data']['input_window_length']

fa = Fasta(FASTA)
chrom_sizes = {c: len(fa[c]) for c in fa.keys()}
print(f'Device: {device} | window={window_size} bp | stride={STRIDE} bp')

In [ ]:
# ── Genome-wide prediction ────────────────────────────────────────────────────
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

def _fetch_window(seq_bytes, ws):
    chunk = seq_bytes[ws: ws + window_size]
    seq = chunk.decode('ascii')
    if len(seq) < window_size:
        seq = seq + 'N' * (window_size - len(seq))
    return one_hot_encode(seq)  # [4, L]

def _predict(batch_oh):
    x = torch.tensor(np.stack(batch_oh), dtype=torch.float32).to(device)
    if RC:
        return rc_average(model, x, SPECIES).cpu().numpy()  # [B, OUT_BINS, 4]
    with torch.no_grad():
        return model(x, head=SPECIES)['rt_signals'].cpu().numpy()

if BINS_CSV is not None:
    print(f'Loading bins from {BINS_CSV}')
    bins_df = pd.read_csv(BINS_CSV)
else:
    print('Running genome-wide prediction...')
    chrom_list, bin_start_list = [], []
    g1_list, es_list, ms_list, ls_list = [], [], [], []

    for chrom in CHROMS:
        clen = chrom_sizes.get(chrom)
        if clen is None:
            continue
        seq_bytes = bytearray(str(fa[chrom]).upper().encode())
        windows   = list(range(0, clen - window_size + 1, STRIDE))

        batch_oh, batch_ws = [], []

        def _flush(batch_oh, batch_ws):
            if not batch_oh:
                return
            preds = _predict(batch_oh)                        # [B, OUT_BINS, 4]
            raw   = np.expm1(np.clip(preds, 0, None))         # undo log1p
            for b, ws in enumerate(batch_ws):
                starts = ws + CROP_BP + np.arange(OUT_BINS) * BIN_SIZE
                valid  = starts + BIN_SIZE <= clen
                n      = valid.sum()
                chrom_list.extend([chrom] * n)
                bin_start_list.extend(starts[valid].tolist())
                g1_list.extend(raw[b, valid, 0].tolist())
                es_list.extend(raw[b, valid, 1].tolist())
                ms_list.extend(raw[b, valid, 2].tolist())
                ls_list.extend(raw[b, valid, 3].tolist())

        for ws in windows:
            batch_oh.append(_fetch_window(seq_bytes, ws))
            batch_ws.append(ws)
            if len(batch_oh) == BATCH_SIZE:
                _flush(batch_oh, batch_ws)
                batch_oh, batch_ws = [], []
        _flush(batch_oh, batch_ws)
        print(f'  {chrom}: done')

    bins_df = pd.DataFrame({
        'chrom':     chrom_list,
        'bin_start': bin_start_list,
        'g1': g1_list, 'es': es_list, 'ms': ms_list, 'ls': ls_list,
    })
    bins_df.to_csv(out_dir / 'bins.csv', index=False)
    print(f'Saved → {out_dir}/bins.csv  (set BINS_CSV to reuse)')

print(f'{len(bins_df):,} bins total')

In [ ]:
# ── Classify bins by G1-normalised phase ─────────────────────────────────────
eps = 1e-6
g1  = bins_df['g1'].values
ratios = np.stack([
    bins_df['es'].values / (g1 + eps),
    bins_df['ms'].values / (g1 + eps),
    bins_df['ls'].values / (g1 + eps),
], axis=1)  # [N, 3]

max_ratio = ratios.max(axis=1)
argmax    = ratios.argmax(axis=1)

bins_df['phase']     = np.where(max_ratio >= 1.0, PHASE_NAMES[argmax], 'non-rep')
bins_df['ratio_es']  = ratios[:, 0]
bins_df['ratio_ms']  = ratios[:, 1]
bins_df['ratio_ls']  = ratios[:, 2]
bins_df['max_ratio'] = max_ratio

print('Genome-wide predicted phase distribution:')
print(bins_df['phase'].value_counts())

## Validation: predicted phase vs. true labels

In [ ]:
# ── Load true labels and apply the same classification rule ──────────────────
# TSV values are log1p(CPM) — undo log1p before computing ratios (same as prediction side)
true_df = load_signals(LABELS_TSV, SPECIES)
true_df = true_df[true_df['chrom'].isin(CHROMS)].copy()
true_df = true_df.rename(columns={'start': 'bin_start'})

true_cpm    = np.expm1(np.clip(true_df[['G1', 'ES', 'MS', 'LS']].values, 0, None))
true_g1     = true_cpm[:, 0]
true_ratios = np.stack([
    true_cpm[:, 1] / (true_g1 + eps),
    true_cpm[:, 2] / (true_g1 + eps),
    true_cpm[:, 3] / (true_g1 + eps),
], axis=1)
true_max    = true_ratios.max(axis=1)
true_df['true_phase'] = np.where(true_max >= 1.0, PHASE_NAMES[true_ratios.argmax(axis=1)], 'non-rep')

# join predicted phase onto true labels
pred_lookup = bins_df.set_index(['chrom', 'bin_start'])['phase'].rename('pred_phase')
true_df = true_df.join(pred_lookup, on=['chrom', 'bin_start'], how='inner')
true_df = true_df.dropna(subset=['pred_phase'])

print(f'Bins with both true label and prediction: {len(true_df):,}')

In [ ]:
# ── Per-chromosome accuracy ───────────────────────────────────────────────────
chrom_acc = {}
for chrom in sorted(true_df['chrom'].unique()):
    sub = true_df[true_df['chrom'] == chrom]
    acc = (sub['pred_phase'] == sub['true_phase']).mean()
    chrom_acc[chrom] = acc

overall_acc = (true_df['pred_phase'] == true_df['true_phase']).mean()

print(f'Overall accuracy: {overall_acc:.4f}  ({len(true_df):,} bins)\n')
print('Per-chromosome accuracy:')
for chrom, acc in sorted(chrom_acc.items()):
    n = (true_df['chrom'] == chrom).sum()
    print(f'  {chrom}: {acc:.4f}  ({n:,} bins)')

# bar chart
fig, ax = plt.subplots(figsize=(10, 4))
chroms_sorted = sorted(chrom_acc.keys())
ax.bar(chroms_sorted, [chrom_acc[c] for c in chroms_sorted], color='#4878d0')
ax.axhline(overall_acc, color='red', linestyle='--', linewidth=1.2,
           label=f'Overall: {overall_acc:.4f}')
ax.set_ylim(0, 1)
ax.set_ylabel('Accuracy')
ax.set_title('Per-chromosome replication phase classification accuracy')
ax.legend()
plt.tight_layout()
plt.savefig(out_dir / 'per_chrom_accuracy.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm      = confusion_matrix(true_df['true_phase'], true_df['pred_phase'], labels=PHASE_ORDER)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)  # row-normalised (recall)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay(cm, display_labels=PHASE_ORDER).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion matrix (counts)')
axes[0].set_xlabel('Predicted phase')
axes[0].set_ylabel('True phase')

ConfusionMatrixDisplay(np.round(cm_norm, 3), display_labels=PHASE_ORDER).plot(
    ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion matrix (row-normalised, recall per class)')
axes[1].set_xlabel('Predicted phase')
axes[1].set_ylabel('True phase')

plt.tight_layout()
plt.savefig(out_dir / 'confusion_matrix.pdf', bbox_inches='tight')
plt.show()

## CPP8 CUT&Tag enrichment analysis

In [ ]:
# ── Load CPP8 CUT&Tag peaks and overlap with bins ────────────────────────────
assert CPP8_BED is not None, "Set CPP8_BED to the path of your CUT&Tag peaks BED file"

peaks_df = pd.read_csv(
    CPP8_BED, sep='\t', header=None, comment='#',
    usecols=[0, 1, 2], names=['chrom', 'peak_start', 'peak_end'],
)
peaks_df = peaks_df[peaks_df['chrom'].isin(CHROMS)].copy()
peaks_df['center']    = (peaks_df['peak_start'] + peaks_df['peak_end']) // 2
peaks_df['bin_start'] = (peaks_df['center'] // BIN_SIZE) * BIN_SIZE

bin_info = bins_df.set_index(['chrom', 'bin_start'])[['phase', 'ratio_es', 'ratio_ms', 'ratio_ls', 'max_ratio']]
peaks_df = peaks_df.join(bin_info, on=['chrom', 'bin_start'], how='left')

n_matched = peaks_df['phase'].notna().sum()
print(f'{len(peaks_df):,} peaks loaded, {n_matched:,} matched to predicted bins')
peaks_df = peaks_df.dropna(subset=['phase'])

In [ ]:
# ── Phase enrichment: bar chart + fold enrichment ────────────────────────────
genome_frac = bins_df['phase'].value_counts(normalize=True).reindex(PHASE_ORDER, fill_value=0)
cpp8_frac   = peaks_df['phase'].value_counts(normalize=True).reindex(PHASE_ORDER, fill_value=0)
fold        = (cpp8_frac + 1e-9) / (genome_frac + 1e-9)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

x, w = np.arange(len(PHASE_ORDER)), 0.35
ax = axes[0]
ax.bar(x - w/2, genome_frac.values, w, label='Genome background', color='#aec7e8')
ax.bar(x + w/2, cpp8_frac.values,   w, label='CPP8 CUT&Tag peaks',
       color=[COLORS[p] for p in PHASE_ORDER], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(PHASE_ORDER)
ax.set_ylabel('Fraction of bins / peaks')
ax.set_title('Replication phase distribution\nCPP8 peaks vs. genome background')
ax.legend()

ax2 = axes[1]
ax2.bar(PHASE_ORDER, fold.values, color=[COLORS[p] for p in PHASE_ORDER])
ax2.axhline(1.0, color='black', linestyle='--', linewidth=0.8)
ax2.set_ylabel('Fold enrichment (CPP8 / genome)')
ax2.set_title('Fold enrichment by replication phase')

plt.tight_layout()
plt.savefig(out_dir / 'cpp8_phase_enrichment.pdf', bbox_inches='tight')
plt.show()

obs_cpp8   = peaks_df['phase'].value_counts().reindex(PHASE_ORDER, fill_value=0).values
obs_genome = bins_df['phase'].value_counts().reindex(PHASE_ORDER, fill_value=0).values
chi2_stat, p_chi2, _, _ = chi2_contingency(np.vstack([obs_cpp8, obs_genome]))
print(f'Chi-square test (phase distribution): χ²={chi2_stat:.2f}, p={p_chi2:.2e}')
print('\nFold enrichment per phase:')
for ph, fe in fold.items():
    print(f'  {ph}: {fe:.3f}x')

In [ ]:
# ── Continuous ES/G1 ratio: CPP8 peaks vs. genome background ─────────────────
rng = np.random.default_rng(42)
bg_idx     = rng.choice(len(bins_df), size=min(len(peaks_df) * 20, len(bins_df)), replace=False)
bg_ratio   = bins_df['ratio_es'].values[bg_idx]
cpp8_ratio = peaks_df['ratio_es'].values

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(bg_ratio,   bins=100, density=True, alpha=0.5, label='Genome background', color='#aec7e8')
ax.hist(cpp8_ratio, bins=100, density=True, alpha=0.75, label='CPP8 CUT&Tag peaks', color='#d62728')
ax.set_xlabel('ES / G1 ratio')
ax.set_ylabel('Density')
ax.set_title('Early S-phase signal at CPP8 peaks vs. genome background')
ax.legend()
plt.tight_layout()
plt.savefig(out_dir / 'cpp8_ratio_es_dist.pdf', bbox_inches='tight')
plt.show()

stat, p_mw = mannwhitneyu(cpp8_ratio, bg_ratio, alternative='greater')
print(f'Mann-Whitney U (CPP8 ES/G1 > background): p={p_mw:.2e}')
print(f'Median ES/G1 — CPP8: {np.median(cpp8_ratio):.4f}  |  background: {np.median(bg_ratio):.4f}')

In [ ]:
# ── Per-phase ratio boxplot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
ratio_cols = [('ratio_es', 'ES/G1', '#d62728'),
              ('ratio_ms', 'MS/G1', '#ff7f0e'),
              ('ratio_ls', 'LS/G1', '#1f77b4')]

for ax, (col, label, color) in zip(axes, ratio_cols):
    bg_vals   = bins_df[col].values[bg_idx]
    cpp8_vals = peaks_df[col].values
    bp = ax.boxplot([bg_vals, cpp8_vals], labels=['Background', 'CPP8'],
                    patch_artist=True, medianprops=dict(color='black', linewidth=1.5))
    bp['boxes'][0].set_facecolor('#aec7e8')
    bp['boxes'][1].set_facecolor(color)
    ax.set_title(label)
    ax.set_ylabel('Signal / G1')
    _, p = mannwhitneyu(cpp8_vals, bg_vals, alternative='two-sided')
    ax.set_xlabel(f'Mann-Whitney p={p:.2e}')

plt.suptitle('CPP8 CUT&Tag peaks — signal ratios vs. genome background', y=1.02)
plt.tight_layout()
plt.savefig(out_dir / 'cpp8_ratio_boxplot.pdf', bbox_inches='tight')
plt.show()